In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    recall_score, precision_score, confusion_matrix
)

In [2]:


# 1. Semilla de reproducibilidad
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 2. Carga y estructuración de dataset.csv (Formato Melt -> Matrices del Grafo)
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep=None, engine= 'python')
# Mapeo ordenado de las 153 bacterias y los 121 pacientes
bact_map = df[['bacteria_id', 'Bacteria_Limpia']].drop_duplicates().sort_values('bacteria_id')
nombres_bacterias = bact_map['Bacteria_Limpia'].values

pat_map = df[['paciente_id', 'Paciente', 'Diagnosticos']].drop_duplicates().sort_values('paciente_id')
y_all = pat_map['Diagnosticos'].values  # 1 = TEA (71 pacientes), 0 = HC (50 pacientes)

# Matriz de Abundancia CLR (121 x 153) y Máscara de Presencia de Aristas Bipartitas
pivot_clr = df.pivot(index='paciente_id', columns='bacteria_id', values='AbundanciaTaxonomica')
X_all = pivot_clr.fillna(0.0).values
M_all = (~pivot_clr.isna()).astype(np.float32).values


# 3. Capa GAT Relacional para un color de cable Bacteria <-> Bacteria
class RelationalBacteriaGATLayer(nn.Module):
    """
    Aplica Graph Attention sobre un tipo específico de cable Bacteria <-> Bacteria
    (Rojo = Red TEA, Verde = Red HC, Violeta = Recableado Diferencial).
    """
    def __init__(self, in_features, out_features, dropout=0.25, alpha=0.2):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a_src = nn.Parameter(torch.empty(size=(out_features, 1)))
        self.a_dst = nn.Parameter(torch.empty(size=(out_features, 1)))
        self.a_edge = nn.Parameter(torch.empty(size=(1, 1)))

        nn.init.xavier_uniform_(self.W.weight, gain=1.414)
        nn.init.xavier_uniform_(self.a_src.data, gain=1.414)
        nn.init.xavier_uniform_(self.a_dst.data, gain=1.414)
        nn.init.constant_(self.a_edge.data, 1.0)

        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h_bact, adj_matrix):
        Wh = self.W(h_bact)  # (153, out_features)
        e_src = torch.matmul(Wh, self.a_src)                  # (153, 1)
        e_dst = torch.matmul(Wh, self.a_dst).transpose(0, 1)  # (1, 153)

        # Atención modulada por la intensidad de correlación del cable
        e = self.leakyrelu(e_src + e_dst + self.a_edge * torch.abs(adj_matrix))

        eye = torch.eye(adj_matrix.size(0), device=adj_matrix.device)
        mask = (torch.abs(adj_matrix) > 0) | (eye > 0)
        e_masked = e.masked_fill(~mask, -1e9)

        attention = F.softmax(e_masked, dim=1)
        attention_drop = self.dropout(attention)

        # Propagación respetando el signo de la correlación (simbiosis vs competencia)
        signed_weights = adj_matrix + eye
        h_prime = torch.matmul(attention_drop * signed_weights, Wh)
        return F.elu(h_prime), attention


# 4. Arquitectura GAT Heterogénea Multirelacional (4 Colores de Cables)
class MultiColorHeteroGAT(nn.Module):
    def __init__(self, bact_in_dim=9, hidden_dim=32, dropout=0.25):
        super().__init__()
        # Fase 1: Cabezales GAT independientes por cada color de cable Bacteria-Bacteria
        self.gat_red = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout)    # Cable Rojo (TEA)
        self.gat_green = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout)  # Cable Verde (HC)
        self.gat_purple = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout) # Cable Violeta (Delta)
        self.bact_self = nn.Linear(bact_in_dim, hidden_dim)

        # Integración multicanal de cada bacteria: [Propia, Roja, Verde, Violeta, Roja - Verde]
        self.bact_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 5, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU()
        )

        # Fase 2: Atención Dirigida Bacteria -> Paciente (Cable Azul de Abundancia CLR)
        self.att_bact = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.att_edge = nn.Linear(3, hidden_dim, bias=False)  # [CLR_norm, Z_score_vs_HC, Presencia]
        self.att_vec = nn.Linear(hidden_dim, 1, bias=False)

        self.msg_proj = nn.Linear(hidden_dim + 3, hidden_dim)

        # Clasificador final de Nodos Paciente (TEA vs HC)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 16),
            nn.ELU(),
            nn.Linear(16, 1)
        )

    def forward(self, bact_features, adj_red, adj_green, adj_purple, edge_feats_pb):
        # 1. Aprendizaje Ecológico Bacteria <-> Bacteria
        h_self = F.elu(self.bact_self(bact_features))
        h_red, att_red = self.gat_red(bact_features, adj_red)
        h_green, att_green = self.gat_green(bact_features, adj_green)
        h_purple, att_purple = self.gat_purple(bact_features, adj_purple)
        h_diff = h_red - h_green  # Contraste directo entre cable rojo y cable verde

        h_bact = self.bact_fusion(
            torch.cat([h_self, h_red, h_green, h_purple, h_diff], dim=-1)
        )  # (153, H)

        # 2. Paso de Mensajes Dirigido Bacteria -> Paciente (Bloquea Paciente -> Bacteria -> Paciente)
        P, B, _ = edge_feats_pb.shape
        h_bact_exp = h_bact.unsqueeze(0).expand(P, B, -1)  # (121, 153, H)

        e_pb = F.leaky_relu(self.att_bact(h_bact_exp) + self.att_edge(edge_feats_pb), negative_slope=0.2)
        scores_pb = self.att_vec(e_pb).squeeze(-1)         # (121, 153)

        alpha_pb = F.softmax(scores_pb, dim=1)             # Atención de cada paciente hacia las 153 bacterias

        clr_norm = edge_feats_pb[:, :, 0:1]
        msg_input = torch.cat([h_bact_exp * clr_norm, edge_feats_pb], dim=-1)
        messages = self.msg_proj(msg_input)                # (121, 153, H)

        # Agregación en cada nodo Paciente
        h_patient = torch.sum(alpha_pb.unsqueeze(-1) * messages, dim=1)  # (121, H)
        logits = self.classifier(h_patient).squeeze(-1)

        return logits, alpha_pb, att_red, att_green, h_diff


# 5. Constructor del Grafo dentro de cada Fold (CERO DATA LEAKAGE)
def build_fold_hetero_graph(X_all, M_all, y_all, train_idx, corr_thresh=0.35, diff_thresh=0.40):
    X_train = X_all[train_idx]
    M_train = M_all[train_idx]
    y_train = y_all[train_idx]

    X_train_tea = X_train[y_train == 1]
    X_train_hc = X_train[y_train == 0]

    # Matrices de co-ocurrencia calculadas SOLO en train_idx
    corr_tea = np.nan_to_num(np.corrcoef(X_train_tea, rowvar=False), 0.0)
    corr_hc = np.nan_to_num(np.corrcoef(X_train_hc, rowvar=False), 0.0)
    np.fill_diagonal(corr_tea, 0.0)
    np.fill_diagonal(corr_hc, 0.0)
    corr_diff = corr_tea - corr_hc

    # Cables Rojos (TEA), Verdes (HC) y Violetas (Recableado)
    adj_red = np.where(np.abs(corr_tea) >= corr_thresh, corr_tea, 0.0)
    adj_green = np.where(np.abs(corr_hc) >= corr_thresh, corr_hc, 0.0)
    adj_purple = np.where(np.abs(corr_diff) >= diff_thresh, corr_diff, 0.0)

    # Características del nodo Bacteria (calculadas solo con train_idx)
    mean_tea = X_train_tea.mean(axis=0)
    mean_hc = X_train_hc.mean(axis=0)
    std_tea = X_train_tea.std(axis=0) + 1e-6
    std_hc = X_train_hc.std(axis=0) + 1e-6
    effect_size = (mean_tea - mean_hc) / np.sqrt(0.5 * (std_tea**2 + std_hc**2))

    prev_tea = M_train[y_train == 1].mean(axis=0)
    prev_hc = M_train[y_train == 0].mean(axis=0)
    deg_red = (np.abs(adj_red) > 0).mean(axis=0)
    deg_green = (np.abs(adj_green) > 0).mean(axis=0)
    deg_purple = (np.abs(adj_purple) > 0).mean(axis=0)

    bact_features = np.stack([
        mean_tea, mean_hc, effect_size,
        prev_tea, prev_hc, prev_tea - prev_hc,
        deg_red, deg_green, deg_purple
    ], axis=1)

    # Cables Azules Bacteria -> Paciente (estandarizados con parámetros de train_idx)
    mean_global = X_train.mean(axis=0, keepdims=True)
    std_global = X_train.std(axis=0, keepdims=True) + 1e-6
    X_norm = (X_all - mean_global) / std_global
    z_from_hc = (X_all - mean_hc[None, :]) / std_hc[None, :]

    edge_feats_pb = np.stack([X_norm, z_from_hc, M_all], axis=-1)

    return (
        torch.tensor(bact_features, dtype=torch.float32),
        torch.tensor(adj_red, dtype=torch.float32),
        torch.tensor(adj_green, dtype=torch.float32),
        torch.tensor(adj_purple, dtype=torch.float32),
        torch.tensor(edge_feats_pb, dtype=torch.float32)
    )


# 6. Ejecución de Validación Cruzada Estratificada (5-Fold CV) y Extracción de Biomarcadores TEA
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_folds = []

# Acumuladores de explicabilidad para niños con TEA
atencion_tea_acum = np.zeros(len(nombres_bacterias))
recableado_acum = np.zeros(len(nombres_bacterias))
gradiente_tea_acum = np.zeros(len(nombres_bacterias))

print("=== ENTRENANDO GAT HETEROGÉNEO MULTIRELACIONAL (5-FOLD CV) ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    set_seed(42 + fold)
    bact_feat, adj_red, adj_green, adj_purple, edge_pb = build_fold_hetero_graph(
        X_all, M_all, y_all, train_idx, corr_thresh=0.35, diff_thresh=0.40
    )
    y_tensor = torch.tensor(y_all, dtype=torch.float32)

    model = MultiColorHeteroGAT(bact_in_dim=9, hidden_dim=32, dropout=0.25)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.006, weight_decay=1.5e-2)

    pos_weight = torch.tensor([(y_all[train_idx] == 0).sum() / (y_all[train_idx] == 1).sum()], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(110):
        model.train()
        optimizer.zero_grad()
        logits, _, _, _, _ = model(bact_feat, adj_red, adj_green, adj_purple, edge_pb)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        optimizer.step()

    # Evaluación en el conjunto de validación (val_idx)
    model.eval()
    edge_pb_grad = edge_pb.clone().detach().requires_grad_(True)
    logits, alpha_pb, att_red, att_green, h_diff = model(bact_feat, adj_red, adj_green, adj_purple, edge_pb_grad)

    # Gradiente de atribución hacia el diagnóstico positivo de TEA (Saliency por bacteria)
    tea_val_nodes = val_idx[y_all[val_idx] == 1]
    score_tea = logits[tea_val_nodes].sum()
    score_tea.backward()

    with torch.no_grad():
        probs = torch.sigmoid(logits[val_idx]).numpy()
        preds = (probs >= 0.5).astype(int)
        y_val = y_all[val_idx]

        acc = accuracy_score(y_val, preds)
        auc = roc_auc_score(y_val, probs)
        f1 = f1_score(y_val, preds)
        sens = recall_score(y_val, preds)
        tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
        spec = tn / (tn + fp)

        metricas_folds.append([fold + 1, acc, auc, f1, sens, spec])
        print(f"Fold {fold+1} | Acc: {acc:.4f} | ROC-AUC: {auc:.4f} | F1: {f1:.4f} | Sens(TEA): {sens:.4f} | Spec(HC): {spec:.4f}")

        # 1. Atención asignada por el GAT específicamente en los niños con TEA
        atencion_tea_acum += alpha_pb[tea_val_nodes].mean(dim=0).numpy() / 5.0
        # 2. Magnitud del recableado entre el Cable Rojo y el Cable Verde
        recableado_acum += torch.norm(h_diff, p=2, dim=1).numpy() / 5.0
        # 3. Impacto directo del cable sobre la predicción de TEA
        grad_bact = edge_pb_grad.grad[tea_val_nodes, :, 0].abs().mean(dim=0).numpy()
        gradiente_tea_acum += grad_bact / 5.0

df_metricas = pd.DataFrame(metricas_folds, columns=['Fold', 'Accuracy', 'ROC_AUC', 'F1_Score', 'Sensibilidad_TEA', 'Especificidad_HC'])
print("\n=== RESUMEN DE VALIDACIÓN CRUZADA ===")
print(df_metricas.mean().to_string())

# 7. Construcción del Ranking Final de Bacterias Críticas para Niños con TEA
mean_clr_tea = X_all[y_all == 1].mean(axis=0)
mean_clr_hc = X_all[y_all == 0].mean(axis=0)
delta_clr = mean_clr_tea - mean_clr_hc

# Normalización Min-Max para combinar Atención + Recableado (Rojo vs Verde) + Gradiente TEA
def norm_minmax(v):
    return (v - v.min()) / (v.max() - v.min() + 1e-8)

score_integral_tea = (
    0.45 * norm_minmax(atencion_tea_acum) +
    0.35 * norm_minmax(gradiente_tea_acum) +
    0.20 * norm_minmax(recableado_acum)
)

df_biomarcadores_tea = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Score_GAT_TEA': score_integral_tea,
    'Atencion_Pacientes_TEA': atencion_tea_acum,
    'Indice_Recableado_Rojo_Verde': recableado_acum,
    'Media_CLR_TEA': mean_clr_tea,
    'Media_CLR_HC': mean_clr_hc,
    'Delta_CLR (TEA-HC)': delta_clr,
    'Perfil_en_TEA': np.where(delta_clr > 0, 'Aumentada en TEA (Cable Rojo/Sobrecrecimiento)', 'Disminuida en TEA (Pérdida de Cable Verde)')
}).sort_values(by='Score_GAT_TEA', ascending=False)

print("\n=== TOP 20 BACTERIAS MÁS IMPORTANTES PARA NIÑOS CON TEA (GAT HETEROGÉNEO) ===")
print(df_biomarcadores_tea.head(20).to_string(index=False))

=== ENTRENANDO GAT HETEROGÉNEO MULTIRELACIONAL (5-FOLD CV) ===
Fold 1 | Acc: 0.6800 | ROC-AUC: 0.6833 | F1: 0.7647 | Sens(TEA): 0.8667 | Spec(HC): 0.4000
Fold 2 | Acc: 0.7500 | ROC-AUC: 0.8857 | F1: 0.8000 | Sens(TEA): 0.8571 | Spec(HC): 0.6000
Fold 3 | Acc: 0.8750 | ROC-AUC: 0.9143 | F1: 0.8966 | Sens(TEA): 0.9286 | Spec(HC): 0.8000
Fold 4 | Acc: 0.8750 | ROC-AUC: 0.9643 | F1: 0.8889 | Sens(TEA): 0.8571 | Spec(HC): 0.9000
Fold 5 | Acc: 0.7917 | ROC-AUC: 0.8821 | F1: 0.8148 | Sens(TEA): 0.7857 | Spec(HC): 0.8000

=== RESUMEN DE VALIDACIÓN CRUZADA ===
Fold                3.000000
Accuracy            0.794333
ROC_AUC             0.865952
F1_Score            0.832992
Sensibilidad_TEA    0.859048
Especificidad_HC    0.700000

=== TOP 20 BACTERIAS MÁS IMPORTANTES PARA NIÑOS CON TEA (GAT HETEROGÉNEO) ===
                    Bacteria  Score_GAT_TEA  Atencion_Pacientes_TEA  Indice_Recableado_Rojo_Verde  Media_CLR_TEA  Media_CLR_HC  Delta_CLR (TEA-HC)                                  Perfil_en_